# Llama 3 8B 4bit 양자화와 속도 사이즈 비교
- Runtime type: A100

In [1]:
!pip -q install -U transformers accelerate bitsandbytes huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 34.0 MB/s eta 0:00:00


In [2]:
from huggingface_hub import login
from getpass import getpass

token = getpass("HF Token 키: ")
login(token=token)

HF Token 키: ··········


## 오리지널 Llama 모델 vs 4bit NF4
- Meta-Llama-3-8B-Instruct: https://huggingface.co/meta-llama/Meta-Llama-3-8B-Instruct

In [3]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

model_id = "meta-llama/Meta-Llama-3-8B-Instruct"
prompt = "RAG(검색 증강 생성)가 뭔지 쉽게 설명해줘."
max_new_tokens = 80

assert torch.cuda.is_available(), "CUDA(GPU)가 필요합니다. 런타임에서 GPU로 바꿔주세요."

# 토크나이저
tokenizer = AutoTokenizer.from_pretrained(model_id, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

def cuda_mem_gb(x):
    return x / (1024**3)

@torch.no_grad()
def run_generate(model, prompt, max_new_tokens=80):
    # 입력은 모델이 올라간 디바이스로
    device = next(model.parameters()).device
    inputs = tokenizer(prompt, return_tensors="pt").to(device)

    # 결정적 비교(가능한 한)
    torch.manual_seed(0)

    # 타이밍 + 메모리 측정
    torch.cuda.reset_peak_memory_stats()
    torch.cuda.synchronize()
    t0 = time.time()

    outputs = model.generate(
        **inputs,
        max_new_tokens=max_new_tokens,
        do_sample=False,          # 비교를 위해 sampling 끔
        temperature=0.0,
        pad_token_id=tokenizer.eos_token_id,
    )

    torch.cuda.synchronize()
    t1 = time.time()
    peak = torch.cuda.max_memory_allocated()

    text = tokenizer.decode(outputs[0], skip_special_tokens=True)
    # 생성 토큰 수(대략)
    gen_tokens = outputs.shape[-1] - inputs["input_ids"].shape[-1]
    dt = t1 - t0
    tok_per_s = gen_tokens / dt if dt > 0 else float("inf")

    return {
        "text": text,
        "seconds": dt,
        "gen_tokens": int(gen_tokens),
        "tok_per_s": tok_per_s,
        "peak_vram_gb": cuda_mem_gb(peak),
    }


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/51.0k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/73.0 [00:00<?, ?B/s]

## 오리지널 Llama-3-8B-Instruct 모델과 Llama 3 8B 4bit 양자화 모델 로드

In [4]:
def load_original(dtype=torch.bfloat16, offload_folder="/content/offload"):
    """
    오리지널(비양자화) 로드.
    - dtype: bf16 추천 (가능하면), 안되면 fp16로 바꾸세요.
    - device_map='auto' + offload_folder로 VRAM 부족 시 일부 CPU/디스크로 내려갑니다.
    """
    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        torch_dtype=dtype,
        device_map="auto",
        offload_folder=offload_folder,   # VRAM 부족 시 도움
        low_cpu_mem_usage=True,
    )
    t1 = time.time()
    return model, (t1 - t0)

def load_4bit_nf4():
    """
    4bit NF4 로드 (bitsandbytes)
    - weight: 4bit
    - compute: fp16 (연산은 fp16으로)
    """
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_compute_dtype=torch.float16,
    )

    t0 = time.time()
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        low_cpu_mem_usage=True,
    )
    t1 = time.time()
    return model, (t1 - t0)


In [5]:
# --------------- 로드 & 비교 ---------------
results = {}

print("=== (A) 4bit NF4 로딩: 추가 처리(양자화 변환/커널 준비)가 필요해서 로딩 시간이 더 길어짐 ===")
model_4bit, load_time_4bit = load_4bit_nf4()
print(f"로드 시간: {load_time_4bit:.2f}s")
print("모델 dtype(참고):", getattr(model_4bit, "dtype", None))
res_4bit = run_generate(model_4bit, prompt, max_new_tokens=max_new_tokens)
results["4bit_nf4"] = {"load_s": load_time_4bit, **res_4bit}
results["4bit_nf4"]["num_params"] = model_4bit.num_parameters()
results["4bit_nf4"]["weight_bits"] = 4

=== (A) 4bit NF4 로딩: 추가 처리(양자화 변환/커널 준비)가 필요해서 로딩 시간이 더 길어짐 ===


config.json:   0%|          | 0.00/654 [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/23.9k [00:00<?, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

model-00002-of-00004.safetensors:   0%|          | 0.00/5.00G [00:00<?, ?B/s]

...

model-00003-of-00004.safetensors:   0%|          | 0.00/4.92G [00:00<?, ?B/s]

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/187 [00:00<?, ?B/s]

The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


로드 시간: 85.33s
모델 dtype(참고): torch.float16


In [6]:
print("\n=== (B) 오리지널(bf16 권장) 로딩 ===")
try:
    model_org, load_time_org = load_original(dtype=torch.bfloat16)
except Exception as e:
    print("bf16 로딩 실패 → fp16로 재시도:", str(e)[:200])
    model_org, load_time_org = load_original(dtype=torch.float16)

print(f"로드 시간: {load_time_org:.2f}s")
print("모델 dtype:", next(model_org.parameters()).dtype)
res_org = run_generate(model_org, prompt, max_new_tokens=max_new_tokens)
results["original"] = {"load_s": load_time_org, **res_org}
results["original"]["num_params"] = model_org.num_parameters()
results["original"]["weight_bits"] = 16


=== (B) 오리지널(bf16 권장) 로딩 ===


`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

로드 시간: 5.98s
모델 dtype: torch.bfloat16


In [7]:
# VRAM 계산
def bytes_to_gb(x):
    return x / (1024**3)

def estimate_weight_vram_gb(num_params: int, bits: int) -> float:
    """모델 weight(파라미터) 저장에 필요한 메모리(이론값)"""
    return bytes_to_gb(num_params * (bits / 8))

# --------------- 요약 출력  ---------------
print("\n\n==================== 비교 요약 ====================")
for k, v in results.items():
    print(f"\n[{k}]")
    print(f"- 모델 로딩 시간       : {v['load_s']:.2f}초")
    print(f"- 생성 토큰 수         : {v['gen_tokens']}개")
    print(f"- 생성(추론) 시간      : {v['seconds']:.2f}초")
    print(f"- 처리 속도(tokens/sec): {v['tok_per_s']:.2f} tokens/s")

    # ✅ 추가: Weight VRAM (이론값)
    if "num_params" in v and "weight_bits" in v:
        weight_est_gb = estimate_weight_vram_gb(v["num_params"], v["weight_bits"])
        print(f"- Weight VRAM(추정)    : {weight_est_gb:.2f}GB  (bits={v['weight_bits']})")

print("\n\n==================== 생성 결과(텍스트) ====================")
print("\n--- [4bit NF4 (양자화)] ---")
print(results["4bit_nf4"]["text"])
print("\n--- [Original (원본)] ---")
print(results["original"]["text"])




==================== 비교 요약 ====================

[4bit_nf4]
- 모델 로딩 시간       : 85.33초
- 생성 토큰 수         : 80개
- 생성(추론) 시간      : 5.09초
- 처리 속도(tokens/sec): 15.71 tokens/s
- Weight VRAM(추정)    : 3.74GB  (bits=4)

[original]
- 모델 로딩 시간       : 5.98초
- 생성 토큰 수         : 80개
- 생성(추론) 시간      : 3.44초
- 처리 속도(tokens/sec): 23.29 tokens/s
- Weight VRAM(추정)    : 14.96GB  (bits=16)


==================== 생성 결과(텍스트) ====================

--- [4bit NF4 (양자화)] ---
RAG(검색 증강 생성)가 뭔지 쉽게 설명해줘. 😊

RAG(Recurrent Attention Generator) is a type of neural network architecture that generates text by combining the strengths of two techniques: Recurrent Neural Networks (RNNs) and Attention Mechanisms.

**RNNs**: RNNs are designed to process sequential data, such as text, by maintaining an internal state that captures the context of the input sequence. This allows RNN

--- [Original (원본)] ---
RAG(검색 증강 생성)가 뭔지 쉽게 설명해줘.  
RAG는 검색 증강 생성(Reinforcement Augmented Generation)입니다.  
이 기술은 기존의 생성 모델에 강화학습(RL: Reinfo

## 토큰 생성 속도(TPS) 벤치마크
- 실제 속도를 측정하여 보여주는 코드

In [8]:
print("\n=== (A) 4bit NF4 토큰 생성 속도(TPS) ===")

def benchmark_speed(model_4bit, tokenizer, prompt, num_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print(f"============ 속도 측정 시작 ({num_tokens} 토큰 생성)...")
    start = time.time()
    _ = model_4bit.generate(
        **inputs,
        max_new_tokens=num_tokens,
        pad_token_id=tokenizer.eos_token_id
    )
    end = time.time()

    duration = end - start
    tps = num_tokens / duration

    print(f" - 소요 시간: {duration:.2f}초")
    print(f" - 처리 속도: {tps:.2f} tokens/sec")
    return tps

# 벤치마크 실행
dummy_prompt = "인터넷의 역사에 관에서 긴 에세이 만들어줘."
benchmark_speed(model_4bit, tokenizer, dummy_prompt, num_tokens=100)


=== (A) 4bit NF4 토큰 생성 속도(TPS) ===
============ 속도 측정 시작 (100 토큰 생성)...
 - 소요 시간: 5.77초
 - 처리 속도: 17.32 tokens/sec


17.324820306224776

In [9]:
print("\n=== (B) 오리지널(bf16) 토큰 생성 속도(TPS) ===")
def benchmark_speed(model_org, tokenizer, prompt, num_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to("cuda")

    print(f"============ 속도 측정 시작 ({num_tokens} 토큰 생성)...")
    start = time.time()
    _ = model_org.generate(
        **inputs,
        max_new_tokens=num_tokens,
        pad_token_id=tokenizer.eos_token_id
    )
    end = time.time()

    duration = end - start
    tps = num_tokens / duration

    print(f" - 소요 시간: {duration:.2f}초")
    print(f" - 처리 속도: {tps:.2f} tokens/sec")
    return tps

# 벤치마크 실행
dummy_prompt = "인터넷의 역사에 관에서 긴 에세이 만들어줘."
benchmark_speed(model_org, tokenizer, dummy_prompt, num_tokens=100)


=== (B) 오리지널(bf16) 토큰 생성 속도(TPS) ===
============ 속도 측정 시작 (100 토큰 생성)...
 - 소요 시간: 3.68초
 - 처리 속도: 27.20 tokens/sec


27.196397315623543

## !!! 영어 프롬프트로 테스트 !!!

- 한국어 프롬프트의 결과가 좋지 않아서 영어 프롬프트로 다시 테스트 합니다.
- LLama의 경우 3.3 이후가 양자화 경우 한국어 결과를 좀 더 낫게 만들어 줍니다.


In [10]:
prompt = "Explain what RAG (Retrieval-Augmented Generation) is in simple terms."

In [14]:
# --------------- 영어 프롬프트 로드 & 비교 ---------------
results = {}

print("=== (A) 4bit NF4 영어 프롬프트로 테스트 ===")

res_4bit = run_generate(model_4bit, prompt, max_new_tokens=max_new_tokens)
results["4bit_nf4"] = {"load_s": load_time_4bit, **res_4bit}
results["4bit_nf4"]["num_params"] = model_4bit.num_parameters()
results["4bit_nf4"]["weight_bits"] = 4

=== (A) 4bit NF4 영어 프롬프트로 테스트 ===


In [15]:
print("\n=== (B) 오리지널(bf16 권장) 영어 프롬프트로 테스트 ===")

res_org = run_generate(model_org, prompt, max_new_tokens=max_new_tokens)
results["original"] = {"load_s": load_time_org, **res_org}
results["original"]["num_params"] = model_org.num_parameters()
results["original"]["weight_bits"] = 16


=== (B) 오리지널(bf16 권장) 영어 프롬프트로 테스트 ===


In [16]:
# --------------- 영어 프롬프트 요약 출력  ---------------
print("\n\n==================== 비교 요약 ====================")
for k, v in results.items():
    print(f"\n[{k}]")
    print(f"- 생성(추론) 시간      : {v['seconds']:.2f}초")
    print(f"- 처리 속도(tokens/sec): {v['tok_per_s']:.2f} tokens/s")

    if "num_params" in v and "weight_bits" in v:
        weight_est_gb = estimate_weight_vram_gb(v["num_params"], v["weight_bits"])
        print(f"- Weight VRAM(추정)    : {weight_est_gb:.2f}GB  (bits={v['weight_bits']})")

print("\n\n==================== 생성 결과(텍스트) ====================")
print("\n--- [4bit NF4 (양자화)] ---")
print(results["4bit_nf4"]["text"])
print("\n--- [Original (원본)] ---")
print(results["original"]["text"])




==================== 비교 요약 ====================

[4bit_nf4]
- 생성(추론) 시간      : 4.09초
- 처리 속도(tokens/sec): 19.54 tokens/s
- Weight VRAM(추정)    : 3.74GB  (bits=4)

[original]
- 생성(추론) 시간      : 2.87초
- 처리 속도(tokens/sec): 27.92 tokens/s
- Weight VRAM(추정)    : 14.96GB  (bits=16)


==================== 생성 결과(텍스트) ====================

--- [4bit NF4 (양자화)] ---
Explain what RAG (Retrieval-Augmented Generation) is in simple terms. How does it differ from other AI models like transformer-based models and traditional recurrent neural networks (RNNs)?

RAG (Retrieval-Augmented Generation) is a type of AI model that combines the strengths of both retrieval-based and generation-based models. In simple terms, RAG models use a combination of pre-trained language models and a retrieval mechanism to generate text.

Here's how it works:



--- [Original (원본)] ---
Explain what RAG (Retrieval-Augmented Generation) is in simple terms. How does it differ from other AI models like GPT-3 and BERT?

RAG (Ret